In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import polars as pl
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder

import warnings
import os
import logging

In [2]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

# Set display width (prevents wrapping)
pd.set_option('display.width', None)

# Set max column width (show full text)
pd.set_option('display.max_colwidth', None)

# Show full dataframe without scientific notation
pd.set_option('display.float_format', '{:.6f}'.format)

# Expand frame representation
pd.set_option('display.expand_frame_repr', False)



# 🔹 Ignore all Python warnings
warnings.filterwarnings("ignore")

# 🔹 Ignore warnings from environment
os.environ["PYTHONWARNINGS"] = "ignore"

# 🔹 Disable logging warnings (for libraries like LightGBM, transformers, etc.)
logging.getLogger().setLevel(logging.ERROR)

# 🔹 Optional: Suppress specific common warnings
warnings.simplefilter("ignore")



!rm -rf /kaggle/working/*

    

%matplotlib inline

In [3]:
train=pd.read_csv("/kaggle/input/competitions/cyber-physical-anomaly-detection-for-der-systems/train.csv", nrows=300000)

test=pd.read_csv("/kaggle/input/competitions/cyber-physical-anomaly-detection-for-der-systems/test.csv")

# print(f"Data Shape: {train.shape}")
# print(f"Checkout null values: {train.isnull().sum()}")
# print("#"*180)

# print(f"Data Shape: {test.shape}")
# print(f"Checkout null values: {test.isnull().sum()}")

In [4]:
train.head()

,Id,common[0].ID,common[0].L,common[0].Mn,common[0].Md,common[0].Opt,common[0].Vr,common[0].SN,common[0].DA,DERMeasureAC[0].ID,DERMeasureAC[0].L,DERMeasureAC[0].ACType,DERMeasureAC[0].W,DERMeasureAC[0].VA,DERMeasureAC[0].Var,DERMeasureAC[0].PF,DERMeasureAC[0].A,DERMeasureAC[0].LLV,DERMeasureAC[0].LNV,DERMeasureAC[0].Hz,DERMeasureAC[0].TotWhInj,DERMeasureAC[0].TotWhAbs,DERMeasureAC[0].TotVarhInj,DERMeasureAC[0].TotVarhAbs,DERMeasureAC[0].TmpAmb,DERMeasureAC[0].TmpCab,DERMeasureAC[0].TmpSnk,DERMeasureAC[0].TmpTrns,DERMeasureAC[0].TmpSw,DERMeasureAC[0].TmpOt,DERMeasureAC[0].WL1,DERMeasureAC[0].VAL1,DERMeasureAC[0].VarL1,DERMeasureAC[0].PFL1,DERMeasureAC[0].AL1,DERMeasureAC[0].VL1L2,DERMeasureAC[0].VL1,DERMeasureAC[0].TotWhInjL1,DERMeasureAC[0].TotWhAbsL1,DERMeasureAC[0].TotVarhInjL1,DERMeasureAC[0].TotVarhAbsL1,DERMeasureAC[0].WL2,DERMeasureAC[0].VAL2,DERMeasureAC[0].VarL2,DERMeasureAC[0].PFL2,DERMeasureAC[0].AL2,DERMeasureAC[0].VL2L3,DERMeasureAC[0].VL2,DERMeasureAC[0].TotWhInjL2,DERMeasureAC[0].TotWhAbsL2,DERMeasureAC[0].TotVarhInjL2,DERMeasureAC[0].TotVarhAbsL2,DERMeasureAC[0].WL3,DERMeasureAC[0].VAL3,DERMeasureAC[0].VarL3,DERMeasureAC[0].PFL3,DERMeasureAC[0].AL3,DERMeasureAC[0].VL3L1,DERMeasureAC[0].VL3,DERMeasureAC[0].TotWhInjL3,DERMeasureAC[0].TotWhAbsL3,DERMeasureAC[0].TotVarhInjL3,DERMeasureAC[0].TotVarhAbsL3,DERMeasureAC[0].ThrotPct,DERMeasureAC[0].ThrotSrc,DERMeasureAC[0].A_SF,DERMeasureAC[0].V_SF,DERMeasureAC[0].Hz_SF,DERMeasureAC[0].W_SF,DERMeasureAC[0].PF_SF,DERMeasureAC[0].VA_SF,DERMeasureAC[0].Var_SF,DERMeasureAC[0].TotWh_SF,DERMeasureAC[0].TotVarh_SF,DERMeasureAC[0].Tmp_SF,DERCapacity[0].ID,DERCapacity[0].L,DERCapacity[0].WMaxRtg,DERCapacity[0].WOvrExtRtg,DERCapacity[0].WOvrExtRtgPF,DERCapacity[0].WUndExtRtg,DERCapacity[0].WUndExtRtgPF,DERCapacity[0].VAMaxRtg,DERCapacity[0].VarMaxInjRtg,DERCapacity[0].VarMaxAbsRtg,DERCapacity[0].WChaRteMaxRtg,DERCapacity[0].WDisChaRteMaxRtg,DERCapacity[0].VAChaRteMaxRtg,DERCapacity[0].VADisChaRteMaxRtg,DERCapacity[0].VNomRtg,DERCapacity[0].VMaxRtg,DERCapacity[0].VMinRtg,DERCapacity[0].AMaxRtg,DERCapacity[0].PFOvrExtRtg,DERCapacity[0].PFUndExtRtg,DERCapacity[0].ReactSusceptRtg,DERCapacity[0].NorOpCatRtg,DERCapacity[0].AbnOpCatRtg,DERCapacity[0].CtrlModes,DERCapacity[0].IntIslandCatRtg,DERCapacity[0].WMax,DERCapacity[0].WMaxOvrExt,DERCapacity[0].WOvrExtPF,DERCapacity[0].WMaxUndExt,DERCapacity[0].WUndExtPF,DERCapacity[0].VAMax,DERCapacity[0].VarMaxInj,DERCapacity[0].VarMaxAbs,DERCapacity[0].WChaRteMax,DERCapacity[0].WDisChaRteMax,DERCapacity[0].VAChaRteMax,DERCapacity[0].VADisChaRteMax,DERCapacity[0].VNom,DERCapacity[0].VMax,DERCapacity[0].VMin,DERCapacity[0].AMax,DERCapacity[0].PFOvrExt,DERCapacity[0].PFUndExt,DERCapacity[0].IntIslandCat,DERCapacity[0].W_SF,DERCapacity[0].PF_SF,DERCapacity[0].VA_SF,DERCapacity[0].Var_SF,DERCapacity[0].V_SF,DERCapacity[0].A_SF,DERCapacity[0].S_SF,DEREnterService[0].ID,DEREnterService[0].L,DEREnterService[0].ES,DEREnterService[0].ESVHi,DEREnterService[0].ESVLo,DEREnterService[0].ESHzHi,DEREnterService[0].ESHzLo,DEREnterService[0].ESDlyTms,DEREnterService[0].ESRndTms,DEREnterService[0].ESRmpTms,DEREnterService[0].ESDlyRemTms,DEREnterService[0].V_SF,DEREnterService[0].Hz_SF,DERCtlAC[0].ID,DERCtlAC[0].L,DERCtlAC[0].PFWInjEna,DERCtlAC[0].PFWInjEnaRvrt,DERCtlAC[0].PFWInjRvrtTms,DERCtlAC[0].PFWInjRvrtRem,DERCtlAC[0].PFWAbsEna,DERCtlAC[0].PFWAbsEnaRvrt,DERCtlAC[0].PFWAbsRvrtTms,DERCtlAC[0].PFWAbsRvrtRem,DERCtlAC[0].WMaxLimPctEna,DERCtlAC[0].WMaxLimPct,DERCtlAC[0].WMaxLimPctRvrt,DERCtlAC[0].WMaxLimPctEnaRvrt,DERCtlAC[0].WMaxLimPctRvrtTms,DERCtlAC[0].WMaxLimPctRvrtRem,DERCtlAC[0].WSetEna,DERCtlAC[0].WSetMod,DERCtlAC[0].WSet,DERCtlAC[0].WSetRvrt,DERCtlAC[0].WSetPct,DERCtlAC[0].WSetPctRvrt,DERCtlAC[0].WSetEnaRvrt,DERCtlAC[0].WSetRvrtTms,DERCtlAC[0].WSetRvrtRem,DERCtlAC[0].VarSetEna,DERCtlAC[0].VarSetMod,DERCtlAC[0].VarSetPri,DERCtlAC[0].VarSet,DERCtlAC[0].VarSetRvrt,DERCtlAC[0].VarSetPct,DERCtlAC[0].VarSetPctRvrt,DERCtlAC[0].VarSetEnaRvrt,DERCtlAC[0].VarSetRvrtT

In [5]:
test.head()

,Id,common[0].ID,common[0].L,common[0].Mn,common[0].Md,common[0].Opt,common[0].Vr,common[0].SN,common[0].DA,DERMeasureAC[0].ID,DERMeasureAC[0].L,DERMeasureAC[0].ACType,DERMeasureAC[0].W,DERMeasureAC[0].VA,DERMeasureAC[0].Var,DERMeasureAC[0].PF,DERMeasureAC[0].A,DERMeasureAC[0].LLV,DERMeasureAC[0].LNV,DERMeasureAC[0].Hz,DERMeasureAC[0].TotWhInj,DERMeasureAC[0].TotWhAbs,DERMeasureAC[0].TotVarhInj,DERMeasureAC[0].TotVarhAbs,DERMeasureAC[0].TmpAmb,DERMeasureAC[0].TmpCab,DERMeasureAC[0].TmpSnk,DERMeasureAC[0].TmpTrns,DERMeasureAC[0].TmpSw,DERMeasureAC[0].TmpOt,DERMeasureAC[0].WL1,DERMeasureAC[0].VAL1,DERMeasureAC[0].VarL1,DERMeasureAC[0].PFL1,DERMeasureAC[0].AL1,DERMeasureAC[0].VL1L2,DERMeasureAC[0].VL1,DERMeasureAC[0].TotWhInjL1,DERMeasureAC[0].TotWhAbsL1,DERMeasureAC[0].TotVarhInjL1,DERMeasureAC[0].TotVarhAbsL1,DERMeasureAC[0].WL2,DERMeasureAC[0].VAL2,DERMeasureAC[0].VarL2,DERMeasureAC[0].PFL2,DERMeasureAC[0].AL2,DERMeasureAC[0].VL2L3,DERMeasureAC[0].VL2,DERMeasureAC[0].TotWhInjL2,DERMeasureAC[0].TotWhAbsL2,DERMeasureAC[0].TotVarhInjL2,DERMeasureAC[0].TotVarhAbsL2,DERMeasureAC[0].WL3,DERMeasureAC[0].VAL3,DERMeasureAC[0].VarL3,DERMeasureAC[0].PFL3,DERMeasureAC[0].AL3,DERMeasureAC[0].VL3L1,DERMeasureAC[0].VL3,DERMeasureAC[0].TotWhInjL3,DERMeasureAC[0].TotWhAbsL3,DERMeasureAC[0].TotVarhInjL3,DERMeasureAC[0].TotVarhAbsL3,DERMeasureAC[0].ThrotPct,DERMeasureAC[0].ThrotSrc,DERMeasureAC[0].A_SF,DERMeasureAC[0].V_SF,DERMeasureAC[0].Hz_SF,DERMeasureAC[0].W_SF,DERMeasureAC[0].PF_SF,DERMeasureAC[0].VA_SF,DERMeasureAC[0].Var_SF,DERMeasureAC[0].TotWh_SF,DERMeasureAC[0].TotVarh_SF,DERMeasureAC[0].Tmp_SF,DERCapacity[0].ID,DERCapacity[0].L,DERCapacity[0].WMaxRtg,DERCapacity[0].WOvrExtRtg,DERCapacity[0].WOvrExtRtgPF,DERCapacity[0].WUndExtRtg,DERCapacity[0].WUndExtRtgPF,DERCapacity[0].VAMaxRtg,DERCapacity[0].VarMaxInjRtg,DERCapacity[0].VarMaxAbsRtg,DERCapacity[0].WChaRteMaxRtg,DERCapacity[0].WDisChaRteMaxRtg,DERCapacity[0].VAChaRteMaxRtg,DERCapacity[0].VADisChaRteMaxRtg,DERCapacity[0].VNomRtg,DERCapacity[0].VMaxRtg,DERCapacity[0].VMinRtg,DERCapacity[0].AMaxRtg,DERCapacity[0].PFOvrExtRtg,DERCapacity[0].PFUndExtRtg,DERCapacity[0].ReactSusceptRtg,DERCapacity[0].NorOpCatRtg,DERCapacity[0].AbnOpCatRtg,DERCapacity[0].CtrlModes,DERCapacity[0].IntIslandCatRtg,DERCapacity[0].WMax,DERCapacity[0].WMaxOvrExt,DERCapacity[0].WOvrExtPF,DERCapacity[0].WMaxUndExt,DERCapacity[0].WUndExtPF,DERCapacity[0].VAMax,DERCapacity[0].VarMaxInj,DERCapacity[0].VarMaxAbs,DERCapacity[0].WChaRteMax,DERCapacity[0].WDisChaRteMax,DERCapacity[0].VAChaRteMax,DERCapacity[0].VADisChaRteMax,DERCapacity[0].VNom,DERCapacity[0].VMax,DERCapacity[0].VMin,DERCapacity[0].AMax,DERCapacity[0].PFOvrExt,DERCapacity[0].PFUndExt,DERCapacity[0].IntIslandCat,DERCapacity[0].W_SF,DERCapacity[0].PF_SF,DERCapacity[0].VA_SF,DERCapacity[0].Var_SF,DERCapacity[0].V_SF,DERCapacity[0].A_SF,DERCapacity[0].S_SF,DEREnterService[0].ID,DEREnterService[0].L,DEREnterService[0].ES,DEREnterService[0].ESVHi,DEREnterService[0].ESVLo,DEREnterService[0].ESHzHi,DEREnterService[0].ESHzLo,DEREnterService[0].ESDlyTms,DEREnterService[0].ESRndTms,DEREnterService[0].ESRmpTms,DEREnterService[0].ESDlyRemTms,DEREnterService[0].V_SF,DEREnterService[0].Hz_SF,DERCtlAC[0].ID,DERCtlAC[0].L,DERCtlAC[0].PFWInjEna,DERCtlAC[0].PFWInjEnaRvrt,DERCtlAC[0].PFWInjRvrtTms,DERCtlAC[0].PFWInjRvrtRem,DERCtlAC[0].PFWAbsEna,DERCtlAC[0].PFWAbsEnaRvrt,DERCtlAC[0].PFWAbsRvrtTms,DERCtlAC[0].PFWAbsRvrtRem,DERCtlAC[0].WMaxLimPctEna,DERCtlAC[0].WMaxLimPct,DERCtlAC[0].WMaxLimPctRvrt,DERCtlAC[0].WMaxLimPctEnaRvrt,DERCtlAC[0].WMaxLimPctRvrtTms,DERCtlAC[0].WMaxLimPctRvrtRem,DERCtlAC[0].WSetEna,DERCtlAC[0].WSetMod,DERCtlAC[0].WSet,DERCtlAC[0].WSetRvrt,DERCtlAC[0].WSetPct,DERCtlAC[0].WSetPctRvrt,DERCtlAC[0].WSetEnaRvrt,DERCtlAC[0].WSetRvrtTms,DERCtlAC[0].WSetRvrtRem,DERCtlAC[0].VarSetEna,DERCtlAC[0].VarSetMod,DERCtlAC[0].VarSetPri,DERCtlAC[0].VarSet,DERCtlAC[0].VarSetRvrt,DERCtlAC[0].VarSetPct,DERCtlAC[0].VarSetPctRvrt,DERCtlAC[0].VarSetEnaRvrt,DERCtlAC[0].VarSetRvrtT

In [6]:
def handle_nulls(df, is_train=True, fill_values=None):
    always_null_thresh = 0.99
    null_pct = df.isnull().mean()
    always_null_cols = null_pct[null_pct >= always_null_thresh].index.tolist()
    
    print(f"Dropping {len(always_null_cols)} always-null columns")
    df = df.drop(columns=always_null_cols, errors='ignore')
    
    high_null_thresh = 0.30
    null_pct_remaining = df.isnull().mean()
    high_null_cols = null_pct_remaining[
        (null_pct_remaining >= high_null_thresh) & 
        (null_pct_remaining < always_null_thresh)
    ].index.tolist()
    
    print(f"Adding null-flag features for {len(high_null_cols)} high-null columns")
    null_flags = pd.concat([df[col].isnull().astype(np.int8).rename(f"{col}_is_null") for col in high_null_cols],axis=1)

    remaining_null_cols = df.columns[df.isnull().any()].tolist()
    remaining_null_cols = [c for c in remaining_null_cols if c != 'Label']

    if is_train:
        fill_values = {}
        for col in remaining_null_cols:
            if df[col].dtype == object or df[col].nunique() < 20:
                fill_val = df[col].mode()[0] if not df[col].mode().empty else 0
            else:
                fill_val = df[col].median()
            fill_values[col] = fill_val
        df = df.fillna(fill_values)
    else:
        df = df.fillna({col: fill_values[col] for col in remaining_null_cols if col in fill_values})

    df = pd.concat([df, null_flags], axis=1).copy()

    print(f"Remaining nulls after imputation: {df.isnull().sum().sum()}")
    return df, fill_values, always_null_cols


label = train['Label'].copy()
train = train.drop(columns=['Label'])

train_clean, fill_values, dropped_cols = handle_nulls(train, is_train=True)
test_clean, _, _ = handle_nulls(test.drop(columns=[c for c in dropped_cols if c in test.columns], errors='ignore'),is_train=False,fill_values=fill_values)

total_nulls_train = train_clean.isnull().sum(axis=1).rename('total_nulls_in_row')
total_nulls_test  = test_clean.isnull().sum(axis=1).rename('total_nulls_in_row')

train_clean = pd.concat([train_clean, label.rename('Label'), total_nulls_train], axis=1).copy()
test_clean  = pd.concat([test_clean, total_nulls_test], axis=1).copy()

print(f"Train shape after cleaning: {train_clean.shape}")
print(f"Test shape after cleaning:  {test_clean.shape}")

Dropping 184 always-null columns
Adding null-flag features for 2 high-null columns
Remaining nulls after imputation: 0
Dropping 0 always-null columns
Adding null-flag features for 2 high-null columns
Remaining nulls after imputation: 0
Train shape after cleaning: (300000, 544)
Test shape after cleaning:  (1012785, 543)


In [7]:
feature_cols = [c for c in train_clean.columns if c not in ['Label', 'total_nulls_in_row', 'Id']]



def clean_col_names(cols):
    return [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in cols]


X_train = train_clean[feature_cols]
y_train = train_clean['Label']

X_test = test_clean[feature_cols]


X_train.columns = clean_col_names(X_train.columns)
X_test.columns  = clean_col_names(X_test.columns)


test_ids = test_clean['Id']

In [8]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")

Categorical columns: ['common_0__Mn', 'common_0__Md', 'common_0__Opt', 'common_0__Vr', 'common_0__SN', 'DERMeasureDC_0__Prt_0__IDStr', 'DERMeasureDC_0__Prt_1__IDStr']


In [9]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    encoders[col] = le

print("All categorical columns encoded.")
print(X_train.dtypes.value_counts())

y_train = y_train.astype(int).reset_index(drop=True)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

Categorical columns: ['common_0__Mn', 'common_0__Md', 'common_0__Opt', 'common_0__Vr', 'common_0__SN', 'DERMeasureDC_0__Prt_0__IDStr', 'DERMeasureDC_0__Prt_1__IDStr']
All categorical columns encoded.
float64    451
int64       88
int8         2
Name: count, dtype: int64


In [10]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score
import numpy as np

skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

oof_lgb  = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n[LightGBM] Fold {fold + 1}")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': 0.05,
        'num_leaves': 32,
        'max_depth': 6,
        'min_data_in_leaf': 20,
        'subsample': 0.9,
        'colsample_bytree': 0.8,
        'reg_lambda': 0.5,
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1,
        'device': 'gpu',
        'gpu_platform_id': 0,
        'gpu_device_id': 0,
        'gpu_use_dp': True  
    }

    train_data = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data, free_raw_data=False)

    model_lgb = lgb.train(
        params=params,
        train_set=train_data,
        num_boost_round=5000,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=True),
            lgb.log_evaluation(period=100)
        ]
    )

    oof_lgb[val_idx] = model_lgb.predict(X_val)
    test_lgb += model_lgb.predict(X_test) / skf.n_splits

    f2 = fbeta_score(y_val, (oof_lgb[val_idx] >= 0.5).astype(int), beta=2)
    print(f"[LightGBM] Fold {fold+1} F2: {f2:.4f}")

lgb_f2 = fbeta_score(y_train, (oof_lgb >= 0.5).astype(int), beta=2)
print(f"\n[LightGBM] Overall OOF F2: {lgb_f2:.4f}")


[LightGBM] Fold 1


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.780088
[200]	valid_0's auc: 0.825703
[300]	valid_0's auc: 0.84058
[400]	valid_0's auc: 0.874913
[500]	valid_0's auc: 0.876717
[600]	valid_0's auc: 0.877637
[700]	valid_0's auc: 0.878697
[800]	valid_0's auc: 0.878994
[900]	valid_0's auc: 0.878804
Early stopping, best iteration is:
[738]	valid_0's auc: 0.879162
[LightGBM] Fold 1 F2: 0.6682

[LightGBM] Fold 2
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.774547
[200]	valid_0's auc: 0.823254
[300]	valid_0's auc: 0.873244
[400]	valid_0's auc: 0.875758
[500]	valid_0's auc: 0.876941
[600]	valid_0's auc: 0.877434
[700]	valid_0's auc: 0.878245
[800]	valid_0's auc: 0.878685
[900]	valid_0's auc: 0.878834
[1000]	valid_0's auc: 0.879451
[1100]	valid_0's auc: 0.879394
[1200]	valid_0's auc: 0.879307
Early stopping, best iteration is:
[1040]	valid_0's auc: 0.879545
[LightGBM] Fold 2 F2: 0.6867

[LightGBM] Fold 3
Training until va

In [11]:
import xgboost as xgb

oof_xgb  = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n[XGBoost] Fold {fold + 1}")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_xgb = xgb.XGBClassifier(
        n_estimators=5000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=0.5,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc',
        early_stopping_rounds=200,
        tree_method='hist',   # GPU
        device='cuda',
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )

    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=100
    )

    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    test_xgb += model_xgb.predict_proba(X_test)[:, 1] / skf.n_splits

    f2 = fbeta_score(y_val, (oof_xgb[val_idx] >= 0.5).astype(int), beta=2)
    print(f"[XGBoost] Fold {fold+1} F2: {f2:.4f}")

xgb_f2 = fbeta_score(y_train, (oof_xgb >= 0.5).astype(int), beta=2)
print(f"\n[XGBoost] Overall OOF F2: {xgb_f2:.4f}")


[XGBoost] Fold 1
[0]	validation_0-auc:0.69422
[100]	validation_0-auc:0.78230
[200]	validation_0-auc:0.82565
[300]	validation_0-auc:0.85961
[400]	validation_0-auc:0.87487
[500]	validation_0-auc:0.87632
[600]	validation_0-auc:0.87733
[700]	validation_0-auc:0.87712
[800]	validation_0-auc:0.87725
[868]	validation_0-auc:0.87734
[XGBoost] Fold 1 F2: 0.6823

[XGBoost] Fold 2
[0]	validation_0-auc:0.68639
[100]	validation_0-auc:0.80964
[200]	validation_0-auc:0.82338
[300]	validation_0-auc:0.87355
[400]	validation_0-auc:0.87544
[500]	validation_0-auc:0.87772
[600]	validation_0-auc:0.87777
[700]	validation_0-auc:0.87838
[800]	validation_0-auc:0.87855
[900]	validation_0-auc:0.87836
[1000]	validation_0-auc:0.87876
[1100]	validation_0-auc:0.87903
[1200]	validation_0-auc:0.87902
[1300]	validation_0-auc:0.87919
[1331]	validation_0-auc:0.87897
[XGBoost] Fold 2 F2: 0.7132

[XGBoost] Fold 3
[0]	validation_0-auc:0.64764
[100]	validation_0-auc:0.79283
[200]	validation_0-auc:0.82289
[300]	validation_0-auc:

In [12]:
from catboost import CatBoostClassifier

oof_cat  = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n[CatBoost] Fold {fold + 1}")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_cat = CatBoostClassifier(
        iterations=5000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=0.5,
        subsample=0.9,
        bootstrap_type='Bernoulli',
        scale_pos_weight=scale_pos_weight,
        eval_metric='AUC',
        early_stopping_rounds=200,
        task_type='GPU',          # GPU
        random_seed=42,
        verbose=100,
    )

    model_cat.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    oof_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
    test_cat += model_cat.predict_proba(X_test)[:, 1] / skf.n_splits

    f2 = fbeta_score(y_val, (oof_cat[val_idx] >= 0.5).astype(int), beta=2)
    print(f"[CatBoost] Fold {fold+1} F2: {f2:.4f}")

cat_f2 = fbeta_score(y_train, (oof_cat >= 0.5).astype(int), beta=2)
print(f"\n[CatBoost] Overall OOF F2: {cat_f2:.4f}")


[CatBoost] Fold 1


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7127854	best: 0.7127854 (0)	total: 234ms	remaining: 19m 27s
100:	test: 0.8748979	best: 0.8748979 (100)	total: 1.46s	remaining: 1m 10s
200:	test: 0.8769661	best: 0.8771010 (194)	total: 2.54s	remaining: 1m
300:	test: 0.8778901	best: 0.8780292 (254)	total: 3.52s	remaining: 54.9s
400:	test: 0.8775055	best: 0.8780292 (254)	total: 4.46s	remaining: 51.2s
bestTest = 0.8780291975
bestIteration = 254
Shrink model to first 255 iterations.
[CatBoost] Fold 1 F2: 0.6146

[CatBoost] Fold 2


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6126942	best: 0.6126942 (0)	total: 16.5ms	remaining: 1m 22s
100:	test: 0.8743091	best: 0.8743248 (99)	total: 1.27s	remaining: 1m 1s
200:	test: 0.8767302	best: 0.8769846 (195)	total: 2.39s	remaining: 57s
300:	test: 0.8782957	best: 0.8783498 (289)	total: 3.48s	remaining: 54.4s
400:	test: 0.8776995	best: 0.8784864 (320)	total: 4.58s	remaining: 52.5s
500:	test: 0.8773983	best: 0.8784864 (320)	total: 5.71s	remaining: 51.2s
bestTest = 0.8784863651
bestIteration = 320
Shrink model to first 321 iterations.
[CatBoost] Fold 2 F2: 0.6154

[CatBoost] Fold 3


...

0:	test: 0.7113387	best: 0.7113387 (0)	total: 16.7ms	remaining: 1m 23s
100:	test: 0.8750036	best: 0.8750036 (100)	total: 1.27s	remaining: 1m 1s
200:	test: 0.8762439	best: 0.8763938 (198)	total: 2.34s	remaining: 55.9s
300:	test: 0.8773537	best: 0.8773537 (300)	total: 3.4s	remaining: 53.1s
400:	test: 0.8773028	best: 0.8777028 (320)	total: 4.45s	remaining: 51s
500:	test: 0.8776040	best: 0.8777028 (320)	total: 5.49s	remaining: 49.3s
bestTest = 0.8777027726
bestIteration = 320
Shrink model to first 321 iterations.
[CatBoost] Fold 5 F2: 0.6168

[CatBoost] Fold 6


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7120370	best: 0.7120370 (0)	total: 17.2ms	remaining: 1m 25s
100:	test: 0.8757048	best: 0.8757138 (99)	total: 1.25s	remaining: 1m
200:	test: 0.8778300	best: 0.8779241 (170)	total: 2.26s	remaining: 54s
300:	test: 0.8782987	best: 0.8786049 (271)	total: 3.25s	remaining: 50.7s
400:	test: 0.8789183	best: 0.8789183 (400)	total: 4.23s	remaining: 48.5s
500:	test: 0.8780895	best: 0.8789959 (402)	total: 5.22s	remaining: 46.8s
600:	test: 0.8782926	best: 0.8789959 (402)	total: 6.26s	remaining: 45.8s
bestTest = 0.8789959252
bestIteration = 402
Shrink model to first 403 iterations.
[CatBoost] Fold 6 F2: 0.6338

[CatBoost] Fold 7


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7108976	best: 0.7108976 (0)	total: 16.1ms	remaining: 1m 20s
100:	test: 0.8719461	best: 0.8719461 (100)	total: 1.19s	remaining: 57.8s
200:	test: 0.8769175	best: 0.8769175 (200)	total: 2.14s	remaining: 51.2s
300:	test: 0.8769148	best: 0.8771375 (260)	total: 3.04s	remaining: 47.4s
400:	test: 0.8768062	best: 0.8773250 (309)	total: 3.94s	remaining: 45.1s
500:	test: 0.8760432	best: 0.8773250 (309)	total: 4.83s	remaining: 43.4s
bestTest = 0.8773249984
bestIteration = 309
Shrink model to first 310 iterations.
[CatBoost] Fold 7 F2: 0.6035

[CatBoost] Overall OOF F2: 0.6341


In [13]:
print("=" * 45)
print(f"  LightGBM  OOF F2: {lgb_f2:.4f}")
print(f"  XGBoost   OOF F2: {xgb_f2:.4f}")
print(f"  CatBoost  OOF F2: {cat_f2:.4f}")
print("=" * 45)

scores  = np.array([lgb_f2, xgb_f2, cat_f2])
weights = scores / scores.sum()
print(f"\nAuto weights: LGB={weights[0]:.3f}, XGB={weights[1]:.3f}, CAT={weights[2]:.3f}")

oof_ensemble  = (weights[0]*oof_lgb + weights[1]*oof_xgb + weights[2]*oof_cat)
test_ensemble = (weights[0]*test_lgb + weights[1]*test_xgb + weights[2]*test_cat)

best_thresh, best_f2_ens = 0.5, 0.0
for thresh in np.arange(0.1, 0.6, 0.01):
    f2 = fbeta_score(y_train, (oof_ensemble >= thresh).astype(int), beta=2)
    if f2 > best_f2_ens:
        best_f2_ens = f2
        best_thresh = thresh

print(f"\nBest Threshold: {best_thresh:.2f}")
print(f"Ensemble OOF F2 @ best threshold: {best_f2_ens:.4f}")

  LightGBM  OOF F2: 0.6899
  XGBoost   OOF F2: 0.6799
  CatBoost  OOF F2: 0.6341

Auto weights: LGB=0.344, XGB=0.339, CAT=0.316

Best Threshold: 0.25
Ensemble OOF F2 @ best threshold: 0.9106


In [14]:
final_preds = (test_ensemble >= best_thresh).astype(int)

submission = pd.DataFrame({'Id':    test_ids.astype(int),'Label': final_preds})
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved! Shape: {submission.shape}")
print(submission['Label'].value_counts())
print(submission.head())


Submission saved! Shape: (1012785, 2)
Label
1    764245
0    248540
Name: count, dtype: int64
        Id  Label
0  3000576      1
1   356192      1
2  1757012      0
3   738645      1
4   915723      1
